# Parsing des Données XML de Maintenance

In [9]:
import xml.etree.ElementTree as ET

#Charge et parse le fichier xml
tree = ET.parse('maintenance_report.xml')
root = tree.getroot()

import xml.etree.ElementTree as ET

# Charger et parser le fichier XML
tree = ET.parse('maintenance_report.xml')
root = tree.getroot()

# Fonction pour extraire les informations clés d'une opération
def parse_operation(operation):
    type_op = operation.find('type').text
    description = operation.find('description').text
    date_op = operation.find('date').text
    temps_passe = float(operation.find('temps_passe').text)
    cout = float(operation.find('cout').text)
    
    pieces = []
    pieces_utilisees = operation.find('pieces_utilisees')
    if pieces_utilisees is not None:
        for piece in pieces_utilisees.findall('piece'):
            nom_piece = piece.find('nom').text
            reference_piece = piece.find('reference').text
            quantite = int(piece.find('quantite').text)
            pieces.append({
                'nom': nom_piece,
                'reference': reference_piece,
                'quantite': quantite
            })
    
    return {
        'type': type_op,
        'description': description,
        'date': date_op,
        'temps_passe': temps_passe,
        'cout': cout,
        'pieces_utilisees': pieces
    }

# Fonction pour extraire les informations clés d'un incident
def parse_incident(incident):
    date_incident = incident.find('date_incident').text
    description = incident.find('description').text
    diagnostic = incident.find('diagnostic').text
    action_corrigee = incident.find('action_corrigée').text
    temps_arret = float(incident.find('temps_arret').text)
    
    return {
        'date_incident': date_incident,
        'description': description,
        'diagnostic': diagnostic,
        'action_corrigee': action_corrigee,
        'temps_arret': temps_arret
    }

# Fonction pour extraire les informations clés des capteurs
def parse_capteur(capteur):
    type_capteur = capteur.get('type')
    date_capteur = capteur.find('date').text
    valeur = float(capteur.find('valeur').text)
    unite = capteur.find('unite').text
    emplacement = capteur.find('emplacement').text
    
    return {
        'type': type_capteur,
        'date': date_capteur,
        'valeur': valeur,
        'unite': unite,
        'emplacement': emplacement
    }

# Liste pour stocker les résultats parsés
helicopteres_data = []

# Parcourir chaque hélicoptère dans le fichier XML
for helicoptere in root.findall('helicoptere'):
    id_helicoptere = helicoptere.get('id')
    modele_helicoptere = helicoptere.get('modele')
    
    # Extraire les données de maintenance
    maintenance = helicoptere.find('maintenance')
    date_maintenance = maintenance.find('date_maintenance').text
    technicien = maintenance.find('technicien')
    nom_technicien = technicien.get('nom')
    prenom_technicien = technicien.get('prenom')
    
    operations = []
    for operation in maintenance.find('operations').findall('operation'):
        operations.append(parse_operation(operation))
    
    # Extraire les données des incidents
    incidents = []
    for incident in helicoptere.find('incidents').findall('incident'):
        incidents.append(parse_incident(incident))
    
    # Extraire les données des capteurs
    capteurs = []
    for capteur in helicoptere.find('capteurs').findall('capteur'):
        capteurs.append(parse_capteur(capteur))
    
    # Regrouper les données pour cet hélicoptère
    helicopteres_data.append({
        'id_helicoptere': id_helicoptere,
        'modele': modele_helicoptere,
        'date_maintenance': date_maintenance,
        'technicien': {
            'nom': nom_technicien,
            'prenom': prenom_technicien
        },
        'operations': operations,
        'incidents': incidents,
        'capteurs': capteurs
    })

print(helicopteres_data)

[{'id_helicoptere': 'H145D2', 'modele': 'H145', 'date_maintenance': '2024-08-20', 'technicien': {'nom': 'Dupont', 'prenom': 'Jean'}, 'operations': [{'type': 'Maintenance préventive', 'description': 'Inspection complète du moteur et remplacement des filtres', 'date': '2024-08-18', 'temps_passe': 2.5, 'cout': 1500.0, 'pieces_utilisees': [{'nom': 'Filtre à air', 'reference': 'FA-145', 'quantite': 1}, {'nom': 'Huile moteur', 'reference': 'HM-320', 'quantite': 5}]}, {'type': 'Réparation corrective', 'description': 'Réparation du système de freinage', 'date': '2024-08-19', 'temps_passe': 3.0, 'cout': 2000.0, 'pieces_utilisees': [{'nom': 'Étrier de frein', 'reference': 'EB-456', 'quantite': 2}]}], 'incidents': [{'date_incident': '2024-08-15', 'description': 'Surveillance du rotor - vibration excessive détectée', 'diagnostic': 'Usure des paliers du rotor principal', 'action_corrigee': 'Remplacement des paliers du rotor', 'temps_arret': 6.5}, {'date_incident': '2024-08-01', 'description': 'Pert

In [23]:
#import mysql.connector
#SQLAlchemy dispose des capacités ORM (Object-Relational Mapping) puissante pour établir la connexion à la base de données, définir les tables en tant que classes Python, et insérer les données de manière élégante et efficace
# Configuration SQLAlchemy 
from sqlalchemy import create_engine

# Connexion à la base de données
engine = create_engine('mysql+pymysql://root:@localhost/bd_new_facturation_maintenance2')

from sqlalchemy import Column, Integer, String, Float, Date, ForeignKey, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship

Base = declarative_base()

# Definition des modèles pour représenter les tables existantes dans la base de données
class Helicoptere(Base):
    __tablename__ = 'helicopteres'
    id_helicoptere = Column(String(50), primary_key=True)
    modele = Column(String(255))
    date_maintenance = Column(Date)
    nom_technicien = Column(String(255))
    prenom_technicien = Column(String(255))
    operations = relationship("Operation", back_populates="helicoptere")
    incidents = relationship("Incident", back_populates="helicoptere")
    capteurs = relationship("Capteur", back_populates="helicoptere")

class Operation(Base):
    __tablename__ = 'operations'
    id_operation = Column(Integer, primary_key=True, autoincrement=True)
    id_helicoptere = Column(String(50), ForeignKey('helicopteres.id_helicoptere'))
    type_operation = Column(String(255))
    description = Column(Text)
    date_operation = Column(Date)
    temps_passe = Column(Float)
    cout = Column(Float)
    pieces_utilisees = relationship("PieceUtilisee", back_populates="operation")
    helicoptere = relationship("Helicoptere", back_populates="operations")

class PieceUtilisee(Base):
    __tablename__ = 'pieces_utilisees'
    id_piece = Column(Integer, primary_key=True, autoincrement=True)
    id_operation = Column(Integer, ForeignKey('operations.id_operation'))
    nom_piece = Column(String(255))
    reference_piece = Column(String(100))
    quantite = Column(Integer)
    operation = relationship("Operation", back_populates="pieces_utilisees")

class Incident(Base):
    __tablename__ = 'incidents'
    id_incident = Column(Integer, primary_key=True, autoincrement=True)
    id_helicoptere = Column(String(50), ForeignKey('helicopteres.id_helicoptere'))
    date_incident = Column(Date)
    description = Column(Text)
    diagnostic = Column(Text)
    action_corrigee = Column(Text)
    temps_arret = Column(Float)
    helicoptere = relationship("Helicoptere", back_populates="incidents")

class Capteur(Base):
    __tablename__ = 'capteurs'
    id_capteur = Column(Integer, primary_key=True, autoincrement=True)
    id_helicoptere = Column(String(50), ForeignKey('helicopteres.id_helicoptere'))
    type_capteur = Column(String(255))
    date_capteur = Column(Date)
    valeur = Column(Float)
    unite = Column(String(50))
    emplacement = Column(String(255))
    helicoptere = relationship("Helicoptere", back_populates="capteurs")

    
# Création d'une session
Session = sessionmaker(bind=engine)
session = Session()

from sqlalchemy.exc import IntegrityError

# Insértion des données parsées dans les tables SQL
for helicoptere_data in helicopteres_data:
    try:
        # Création d'un objet Helicoptere
        helicoptere = Helicoptere(
            id_helicoptere=helicoptere_data['id_helicoptere'],
            modele=helicoptere_data['modele'],
            date_maintenance=helicoptere_data['date_maintenance'],
            nom_technicien=helicoptere_data['technicien']['nom'],
            prenom_technicien=helicoptere_data['technicien']['prenom']
        )
        
        # Ajout des opérations
        for operation_data in helicoptere_data['operations']:
            operation = Operation(
                id_helicoptere=helicoptere_data['id_helicoptere'],
                type_operation=operation_data['type'],
                description=operation_data['description'],
                date_operation=operation_data['date'],
                temps_passe=operation_data['temps_passe'],
                cout=operation_data['cout']
            )
            session.add(operation)
            session.flush()  # Pour obtenir l'id_operation avant d'ajouter les pièces utilisées

            # Ajout des pièces utilisées pour cette opération
            for piece_data in operation_data['pieces_utilisees']:
                piece = PieceUtilisee(
                    id_operation=operation.id_operation,
                    nom_piece=piece_data['nom'],
                    reference_piece=piece_data['reference'],
                    quantite=piece_data['quantite']
                )
                session.add(piece)
        
        # Ajout des incidents
        for incident_data in helicoptere_data['incidents']:
            incident = Incident(
                id_helicoptere=helicoptere_data['id_helicoptere'],
                date_incident=incident_data['date_incident'],
                description=incident_data['description'],
                diagnostic=incident_data['diagnostic'],
                action_corrigee=incident_data['action_corrigee'],
                temps_arret=incident_data['temps_arret']
            )
            session.add(incident)
        
        # Ajout des capteurs
        for capteur_data in helicoptere_data['capteurs']:
            capteur = Capteur(
                id_helicoptere=helicoptere_data['id_helicoptere'],
                type_capteur=capteur_data['type'],
                date_capteur=capteur_data['date'],
                valeur=capteur_data['valeur'],
                unite=capteur_data['unite'],
                emplacement=capteur_data['emplacement']
            )
            session.add(capteur)
        
        # Ajout de l'hélicoptère à la session
        session.add(helicoptere)

        # Commit des transactions
        session.commit()
        print(f"Les données pour l'hélicoptère {helicoptere_data['id_helicoptere']} ont été insérées avec succès.")
    
    except IntegrityError:
        # Si une erreur se produit On annule la transaction en cours pour éviter d'insérer des données partielles ou corrompues.
        session.rollback()
        print(f"Erreur : Une tentative d'insérer une valeur dupliquée pour l'hélicoptère {helicoptere_data['id_helicoptere']}. L'insertion a été annulée.")
    
    except Exception as e:
        # Gestion des autres exceptions possibles
        session.rollback()
        print(f"Une erreur inattendue s'est produite : {str(e)}")

# Fermer la session
session.close()

print("L'opération d'insertion est terminée.")



Erreur : Une tentative d'insérer une valeur dupliquée pour l'hélicoptère H145D2. L'insertion a été annulée.
Erreur : Une tentative d'insérer une valeur dupliquée pour l'hélicoptère AS350B3. L'insertion a été annulée.
L'opération d'insertion est terminée.
